# Módulo 08 · Lista de Exercícios

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Este módulo resolveu a dor mais antiga da Aurora: *"na minha máquina funciona"*.

## Como usar

| | |
|---|---|
| 🟢 **Aquecimento** | 1–10 · uma ideia por exercício |
| 🟡 **Construção** | 11–24 · combinam conceitos |
| 🔴 **Integração** | 25–36 · perto de produção |
| 🏗️ **Projeto** | O Atlas containerizado |

**Regras de casa:**

1. Os exercícios de namespace (1–6) rodam aqui mesmo. Os de `docker` exigem Docker instalado — **instale**. Ler sobre container não substitui rodar um.
2. Todo 🔴 tem uma armadilha. Encontre-a antes de ler a dica.
3. Escreva o Dockerfile errado **de propósito** ao menos uma vez, e veja o analisador reclamar. Errar de propósito é mais barato do que errar em produção.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 08
# ═══════════════════════════════════════════════════════════════
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

E_LINUX = platform.system() == "Linux"
TEM_DOCKER = shutil.which("docker") is not None
DOCKER_LIGADO = False
if TEM_DOCKER:
    DOCKER_LIGADO = subprocess.run(["docker", "info"], capture_output=True).returncode == 0


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


_garantir("pyyaml", "yaml")
import yaml

print(f"sistema : {platform.system()}")
print(f"docker  : {'🐳 disponível e ligado' if DOCKER_LIGADO else ('instalado, mas o daemon não responde' if TEM_DOCKER else 'não instalado')}")


# ═══════════════════════════════════════════════════════════════
#  Executar comandos
# ═══════════════════════════════════════════════════════════════

def sh(comando: str, mostrar: bool = True, cwd=None, timeout: int = 120) -> str:
    """Roda um comando de shell e devolve a saída."""
    processo = subprocess.run(comando, shell=True, capture_output=True,
                              text=True, cwd=cwd, timeout=timeout)
    saida = (processo.stdout + processo.stderr).rstrip()
    if mostrar and saida:
        print(saida)
    return saida


def docker(comando: str, esperado: str | None = None, mostrar: bool = True) -> str:
    """Executa `docker ...` se houver daemon; senão, mostra o comando.

    💭 Por que este modo duplo?

       Um daemon Docker não roda dentro de todo ambiente (nem dentro de
       um container, nem em CI restrito, nem neste avaliador). Em vez de
       fingir que rodou, o notebook é HONESTO: quando não há daemon, ele
       mostra o comando e a saída típica, marcada como referência.

       🔴 Saída marcada `[referência]` NÃO foi executada. Rode você
          mesmo no terminal — é assim que se aprende Docker.
    """
    linha = f"docker {comando}"
    if DOCKER_LIGADO:
        print(f"$ {linha}")
        return sh(linha, mostrar=mostrar)
    print(f"$ {linha}")
    if esperado:
        for l in textwrap.dedent(esperado).strip("\n").splitlines():
            print(f"  {l}")
    print("  ── [referência] o daemon Docker não está disponível aqui ──")
    return esperado or ""


# ═══════════════════════════════════════════════════════════════
#  Pasta de trabalho
# ═══════════════════════════════════════════════════════════════
def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def arvore(raiz: Path, prefixo: str = ""):
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".git"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        marca = "└── " if ultimo else "├── "
        tamanho = f"  ({item.stat().st_size} B)" if item.is_file() else ""
        print(f"{prefixo}{marca}{item.name}{tamanho}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]):
    """Tabela ASCII alinhada.

    ⚠️ Marcadores ASCII, não emoji: `len("⚠️")` é 2 mas o terminal
       desenha 1 coluna, e a tabela sai torta. Você já viu isso no M03.
    """
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("✅ `sh()`, `docker()`, `preparar()`, `arvore()` e `tabela()` prontos")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Ferramentas das aulas anteriores, reunidas
# ═══════════════════════════════════════════════════════════════
import fnmatch
import re
from collections import defaultdict

INSTRUCOES_QUE_CRIAM_CAMADA = {"FROM", "RUN", "COPY", "ADD", "WORKDIR"}
SEGREDO = re.compile(
    r"(SECRET|PASSWORD|SENHA|TOKEN|API_?KEY|PRIVATE_KEY|CREDENTIAL)"
    r"\w*\s*[= ]\s*[\"']?[^\s\"']{6,}", re.I)
SEGREDO_CHAVE = re.compile(r"(PASSWORD|SENHA|SECRET|TOKEN|API_?KEY|CREDENTIAL)", re.I)
REFERENCIA = re.compile(r"^\$\{?[A-Z_][A-Z0-9_]*(:-[^}]*)?\}?$")
COM_ESTADO = ("postgres", "mysql", "mariadb", "mongo", "elasticsearch",
              "rabbitmq", "clickhouse", "cassandra")


def ler_dockerfile(texto: str):
    passos, buffer, inicio = [], "", 0
    for n, bruta in enumerate(texto.splitlines(), 1):
        s = bruta.strip()
        if (not s or s.startswith("#")) and not buffer:
            continue
        if buffer:
            buffer += " " + s.rstrip("\\").strip()
        else:
            inicio, buffer = n, s.rstrip("\\").strip()
        if bruta.rstrip().endswith("\\"):
            continue
        if buffer:
            partes = buffer.split(None, 1)
            passos.append((inicio, partes[0].upper(),
                           partes[1] if len(partes) > 1 else ""))
        buffer = ""
    return passos


def analisar_dockerfile(texto: str):
    passos = ler_dockerfile(texto)
    achados = []

    def add(g, n, msg, dica=""):
        achados.append((g, n, msg, dica))

    froms = [p for p in passos if p[1] == "FROM"]
    tem_user = any(p[1] == "USER" for p in passos)

    for n, instr, arg in passos:
        if instr == "FROM":
            img = re.split(r"\s+as\s+", arg, flags=re.I)[0].strip()
            if "@sha256:" not in img:
                if ":" not in img.split("/")[-1]:
                    add("🔴", n, f"FROM sem tag: {img}", "fixe a versão")
                elif img.endswith(":latest"):
                    add("🔴", n, f"FROM :latest — {img}", "fixe a versão")
        if instr == "RUN":
            if "apt-get update" in arg and "apt-get install" not in arg:
                add("🔴", n, "apt-get update num RUN separado", "junte update+install")
            if "apt-get install" in arg and "--no-install-recommends" not in arg:
                add("⚠️ ", n, "apt-get install sem --no-install-recommends", "")
            if "apt-get install" in arg and "rm -rf /var/lib/apt/lists" not in arg:
                add("⚠️ ", n, "apt sem limpar /var/lib/apt/lists", "limpe na MESMA camada")
            if re.search(r"\bpip install\b", arg) and "--no-cache-dir" not in arg:
                add("⚠️ ", n, "pip install sem --no-cache-dir", "")
            if SEGREDO.search(arg):
                add("🔴", n, "possível SEGREDO num RUN", "fica no histórico")
        if instr == "ENV" and SEGREDO.search(arg):
            add("🔴", n, "SEGREDO em ENV", "docker inspect revela")
        if instr == "ARG" and SEGREDO.search(arg):
            add("🔴", n, "SEGREDO em ARG", "docker history revela")
        if instr == "ADD" and not arg.startswith(("http://", "https://")):
            add("⚠️ ", n, "ADD com arquivo local", "use COPY")
        if instr in ("CMD", "ENTRYPOINT"):
            if not arg.strip().startswith("["):
                add("🔴", n, f"{instr} em forma de shell", "SIGTERM não chega")
            if "127.0.0.1" in arg or "localhost" in arg:
                add("🔴", n, "escuta em 127.0.0.1", "use 0.0.0.0")

    if not tem_user:
        add("🔴", 0, "nenhum USER: roda como root", "crie um usuário")
    if not any(p[1] in ("CMD", "ENTRYPOINT") for p in passos):
        add("🔴", 0, "nem CMD nem ENTRYPOINT", "")
    if len(froms) == 1 and any(("gcc" in a or "build-essential" in a)
                               for _, i, a in passos if i == "RUN"):
        add("⚠️ ", 0, "compilador na imagem final", "use multi-stage")

    i_dep = next((k for k, (n, i, a) in enumerate(passos)
                  if i in ("COPY", "ADD")
                  and re.search(r"requirements|pyproject|poetry\.lock|package\.json", a)),
                 None)
    i_tudo = next((k for k, (n, i, a) in enumerate(passos)
                   if i == "COPY" and re.match(r"^\.?\s+", a)), None)
    if i_dep is not None and i_tudo is not None and i_tudo < i_dep:
        add("🔴", passos[i_tudo][0], "COPY . antes das dependências", "inverta")
    elif i_dep is None and i_tudo is not None:
        add("⚠️ ", passos[i_tudo][0], "sem etapa de dependências separada", "")
    return passos, achados


def _env(valor):
    if valor is None:
        return {}
    if isinstance(valor, dict):
        return {str(k): str(v) for k, v in valor.items() if v is not None}
    saida = {}
    for item in valor:
        chave, _, v = str(item).partition("=")
        saida[chave] = v
    return saida


def validar_compose(texto: str):
    achados = []

    def add(g, svc, msg, dica=""):
        achados.append((g, svc, msg, dica))

    try:
        doc = yaml.safe_load(texto)
    except yaml.YAMLError as erro:
        return None, [("🔴", "-", f"YAML inválido: {str(erro)[:60]}", "")]
    if not isinstance(doc, dict):
        return None, [("🔴", "-", "não é um mapeamento YAML", "")]
    if "version" in doc:
        add("⚠️ ", "-", "chave `version` obsoleta", "o Compose v2 ignora")
    servicos = doc.get("services") or {}
    if not servicos:
        return doc, [("🔴", "-", "nenhum serviço definido", "")]

    portas_host = defaultdict(list)
    for nome, s in servicos.items():
        s = s or {}
        img = str(s.get("image", ""))
        tem_build = "build" in s
        if img:
            base = img.split("/")[-1]
            if ":" not in base:
                add("🔴", nome, f"imagem sem tag: {img}", "fixe a versão")
            elif img.endswith(":latest"):
                add("🔴", nome, f"imagem :latest — {img}", "fixe a versão")
        elif not tem_build:
            add("🔴", nome, "sem `image` nem `build`", "")
        for chave, valor in _env(s.get("environment")).items():
            if SEGREDO_CHAVE.search(chave) and valor and not REFERENCIA.match(valor.strip()):
                add("🔴", nome, f"segredo literal: {chave}", "use ${VAR} + .env")
        for p in (s.get("ports") or []):
            texto_p = str(p)
            partes = texto_p.split(":")
            so_loopback = len(partes) == 3 and partes[0] in ("127.0.0.1", "localhost")
            if len(partes) >= 2:
                portas_host[partes[-2]].append(nome)
                if any(b in img for b in COM_ESTADO) and not so_loopback:
                    add("🔴", nome, f"banco exposto na rede: {texto_p}",
                        "amarre ao loopback ou remova")
                elif len(partes) == 2:
                    add("⚠️ ", nome, f"porta em todas as interfaces: {texto_p}", "")
            else:
                add("⚠️ ", nome, f"porta sem mapeamento: {texto_p}", "")
        if any(b in img for b in COM_ESTADO) and not s.get("volumes"):
            add("🔴", nome, "serviço com estado SEM volume", "os dados somem")
        dep = s.get("depends_on")
        if isinstance(dep, list):
            add("⚠️ ", nome, "depends_on em lista", "use service_healthy")
        elif isinstance(dep, dict):
            for alvo, cond in dep.items():
                c = (cond or {}).get("condition") if isinstance(cond, dict) else None
                if c == "service_healthy" and not (servicos.get(alvo) or {}).get("healthcheck"):
                    add("🔴", nome, f"espera `{alvo}` saudável, mas ele não tem healthcheck", "")
                elif c == "service_started":
                    add("⚠️ ", nome, f"depends_on {alvo}: service_started", "iniciado ≠ pronto")
        if "restart" not in s and "profiles" not in s:
            add("⚠️ ", nome, "sem política de `restart`", "use unless-stopped")
        if "container_name" in s:
            add("⚠️ ", nome, "container_name fixo", "impede --scale")
        if tem_build and "user" not in s:
            add("⚠️ ", nome, "sem `user`", "confirme o USER no Dockerfile")

    for porta, donos in portas_host.items():
        if len(donos) > 1:
            add("🔴", "-", f"porta {porta} disputada por {donos}", "o up vai falhar")

    grafo = {n: list((s or {}).get("depends_on") or []) for n, s in servicos.items()}
    estado = {}

    def visitar(n, caminho):
        if estado.get(n) == "ok":
            return
        if estado.get(n) == "visitando":
            add("🔴", "-", f"ciclo em depends_on: {' → '.join(caminho + [n])}", "")
            return
        estado[n] = "visitando"
        for viz in grafo.get(n, []):
            if viz in grafo:
                visitar(viz, caminho + [n])
        estado[n] = "ok"

    for n in grafo:
        visitar(n, [])
    return doc, achados


def mesclar(base: dict, sobre: dict) -> dict:
    saida = dict(base)
    for chave, valor in (sobre or {}).items():
        if isinstance(valor, dict) and isinstance(saida.get(chave), dict):
            saida[chave] = mesclar(saida[chave], valor)
        else:
            saida[chave] = valor
    return saida


def mostrar_achados(titulo: str, achados) -> int:
    graves = sum(1 for g, *_ in achados if g == "🔴")
    print(f"── {titulo} — {len(achados)} achados ({graves} graves) ──")
    for g, a, msg, dica in achados:
        alvo = f"L{a}" if isinstance(a, int) and a else (str(a) if a != 0 else "  ")
        print(f"  {g} [{alvo:<9}] {msg}")
    if not achados:
        print("  ✅ nenhum problema")
    return graves


print("✅ `analisar_dockerfile()`, `validar_compose()`, `mesclar()` prontos")

---

# 🟢 Aquecimento (1–10)

### 1 · User namespace

Rode `unshare --user --map-root-user id` e explique, num comentário, por que você virou `root` sem digitar senha nenhuma.

In [ ]:
# 1

### 2 · PID namespace

Crie um namespace de PID e liste os processos dentro dele. Explique por que o seu processo é o PID 1 — e qual responsabilidade isso traz.

In [ ]:
# 2

### 3 · 🔴 Mount namespace

Monte um `tmpfs`, escreva um arquivo dentro, e prove que ele não existe fora do namespace. Relacione com a camada gravável de um container.

In [ ]:
# 3

### 4 · Network namespace

Liste as interfaces dentro e fora. Explique por que dois containers podem escutar na porta 8000 ao mesmo tempo.

In [ ]:
# 4

### 5 · Os cinco juntos

Combine `--user --pid --mount --net --uts` num comando só e descreva, linha a linha, o que você construiu.

In [ ]:
# 5

### 6 · cgroups

Leia `/sys/fs/cgroup/memory.max` e `cpu.max`. Se disserem `max`, explique o que isso significa e como um container mudaria esses valores.

In [ ]:
# 6

### 7 · Contando camadas

Conte à mão as camadas de um Dockerfile, depois confira com `ler_dockerfile()`. Explique por que `ENV` não conta.

In [ ]:
# 7

### 8 · O analisador

Rode `analisar_dockerfile()` num Dockerfile propositalmente ruim e conserte os achados um a um.

In [ ]:
# 8

### 9 · O validador

Escreva um `docker-compose.yml` com três problemas e mostre o validador encontrando os três.

In [ ]:
# 9

### 10 · Primeiro container de verdade

Instale o Docker e rode `docker run --rm hello-world`. Explique cada linha da saída.

In [ ]:
# 10

---

# 🟡 Construção (11–24)

### 11 · 🔴 Apagar não diminui

`COPY` um arquivo grande e `RUN rm` na linha seguinte. Explique por que a imagem não encolhe e como isso vaza segredo.

In [ ]:
# 11

### 12 · Cache: três cenários

Use um simulador (ou o `docker build` real) para comparar: mudou o código, mudou a dependência, mudou o `Dockerfile`.

In [ ]:
# 12

### 13 · Reordenar

Pegue um Dockerfile com `COPY .` cedo demais e reordene. Meça o ganho.

In [ ]:
# 13

### 14 · `.dockerignore`

Escreva um para o seu `projeto_Atlas` e calcule a redução do contexto.

In [ ]:
# 14

### 15 · 🔴 `.gitignore` não basta

Monte um caso em que `.env` está no `.gitignore` e mesmo assim entra na imagem.

In [ ]:
# 15

### 16 · Multi-stage

Converta um Dockerfile de estágio único em multi-stage. Compare os tamanhos.

In [ ]:
# 16

### 17 · Escolha da base

Compare `python:3.12`, `slim` e `alpine` para o Atlas. Justifique por escrito — inclusive por que Alpine é armadilha com Python.

In [ ]:
# 17

### 18 · 🔴 Segredos

Encontre as três formas erradas de passar segredo e escreva a correção de cada uma.

In [ ]:
# 18

### 19 · Não-root

Adicione usuário sem privilégio. Explique por que o `USER` vai depois dos `RUN` de instalação.

In [ ]:
# 19

### 20 · 🔴 `SIGTERM`

Escreva um script que capture `SIGTERM` e registre no log. Rode-o com `CMD` shell e com `CMD` JSON. Meça o tempo de `docker stop` nos dois casos.

In [ ]:
# 20

### 21 · Rede e DNS

Crie uma rede, suba dois containers e faça um alcançar o outro pelo nome. Depois tente com `localhost` e explique o erro.

In [ ]:
# 21

### 22 · 🔴 `depends_on` mente

Monte um compose com um banco lento e uma API que tenta conectar na hora. Provoque a falha e depois corrija com `healthcheck`.

In [ ]:
# 22

### 23 · Healthchecks

Escreva um para Postgres, Mongo, Redis e a API. Justifique o `start_period` de cada um.

In [ ]:
# 23

### 24 · Volumes

Suba um Postgres sem volume, crie uma tabela, remova o container e recrie. Mostre que sumiu. Refaça com volume.

In [ ]:
# 24

---

# 🔴 Integração (25–36)

### 25 · Dockerfile completo do Atlas

Multi-stage, não-root, healthcheck, `CMD` em JSON, escutando em `0.0.0.0`. Passe no `analisar_dockerfile()` com zero graves.

In [ ]:
# 25

### 26 · Compose completo

API + Postgres + Mongo + Redis, com healthcheck em todos e `service_healthy` nas dependências. Zero graves no validador.

In [ ]:
# 26

### 27 · Override de desenvolvimento

Bind mount de código, `--reload`, Adminer. Valide o resultado **mesclado**, não o fragmento.

In [ ]:
# 27

### 28 · Migrações na ordem

Use `service_completed_successfully` para o Alembic rodar antes da API subir.

In [ ]:
# 28

### 29 · 🔴 Variável silenciosa

Rode `docker compose config` com uma variável não definida e mostre a substituição por vazio. Corrija com `${VAR:?erro}`.

In [ ]:
# 29

### 30 · Log para stdout

Ajuste o `observabilidade.py` do M04 para escrever no stdout em vez de arquivo. Prove com `docker compose logs`.

In [ ]:
# 30

### 31 · 🔴 `OOMKilled`

Suba um container com `--memory=64m` e um script que aloca 200 MB. Mostre o `Exited (137)` e confirme com `docker inspect`.

In [ ]:
# 31

### 32 · Limites de recurso

Defina `cpus` e `memory` para cada serviço do Atlas. Justifique os números.

In [ ]:
# 32

### 33 · Backup de volume

Escreva um script que faça backup e restauração do volume do Postgres. Teste a restauração de verdade — backup não testado não é backup.

In [ ]:
# 33

### 34 · Estenda o analisador

Adicione três verificações ao `analisar_dockerfile()`. Sugestões: ausência de `HEALTHCHECK`, `WORKDIR` relativo, `EXPOSE` em porta privilegiada.

In [ ]:
# 34

### 35 · Estenda o validador

Adicione três ao `validar_compose()`. Sugestões: `network_mode: host`, volume anônimo, serviço do qual alguém depende sem `healthcheck`.

In [ ]:
# 35

### 36 · 🔴 Portão de CI

Escreva um script que rode as duas auditorias e saia com código 1 se houver qualquer 🔴. É o que você vai plugar no GitHub Actions no M09.

In [ ]:
# 36

---

# 🏗️ Projeto — O Atlas containerizado

## O contexto

> *"A desenvolvedora nova levou dois dias para rodar o Atlas."*

Depois deste projeto, a resposta é:

```bash
git clone ...
cp .env.example .env      # e preencher
docker compose up -d
```

**Dois comandos e um arquivo.** Em qualquer sistema operacional.

## O que entregar

```
projeto_Atlas/
├── Dockerfile                    ← multi-stage, não-root, healthcheck
├── .dockerignore                 ← 🔴 primeira linha: .env
├── docker-compose.yml            ← API + Postgres + Mongo + Redis
├── docker-compose.override.yml   ← desenvolvimento (bind mount, reload)
├── docker-compose.prod.yml       ← produção (sem reload, com limites)
├── scripts/
│   ├── entrada.sh                ← espera dependências, roda migração
│   └── auditar_containers.py     ← o portão de CI
└── docs/CONTAINERS.md            ← as decisões
```

## Requisitos obrigatórios

| # | Requisito | Pronto quando |
|---|-----------|---------------|
| 1 | Multi-stage | O compilador não está na imagem final |
| 2 | Imagem < 250 MB | `docker images` confirma |
| 3 | 🔴 Não-root | `docker exec ... id` não devolve `uid=0` |
| 4 | 🔴 Nenhum segredo na imagem | `docker history` limpo |
| 5 | `.dockerignore` com `.env` | primeira linha |
| 6 | `CMD` em JSON, escutando `0.0.0.0` | `docker stop` leva < 2 s |
| 7 | Healthcheck em todos os serviços | `docker compose ps` mostra `healthy` |
| 8 | 🔴 `service_healthy` nas dependências | Nenhum `connection refused` na subida |
| 9 | Volumes nos serviços com estado | `down` + `up` preserva os dados |
| 10 | Log no stdout | `docker compose logs` mostra tudo |
| 11 | Zero 🔴 nas auditorias | O portão de CI sai com 0 |

> 📋 **O roteiro passo a passo está em `projeto_Atlas/ROTEIRO_M08.md`.**

## 🧪 Bateria de aceitação

Aponte para o seu projeto e faça tudo passar.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Aponte para o SEU projeto
# ═══════════════════════════════════════════════════════════════
#
#   PROJETO = Path(r"C:\...\Roadmap\projeto_Atlas")

PROJETO = None          # ← troque pelo caminho do seu projeto_Atlas

RESULTADOS = []


def checar(nome: str, condicao: bool, detalhe: str = "") -> bool:
    RESULTADOS.append((nome, bool(condicao)))
    print(f"   {'✅' if condicao else '🔴'} {nome}{('  — ' + detalhe) if detalhe else ''}")
    return bool(condicao)


def placar():
    passou = sum(1 for _, ok in RESULTADOS if ok)
    print(f"\n{'═' * 56}\n  {passou}/{len(RESULTADOS)} verificações passaram")
    if RESULTADOS and passou == len(RESULTADOS):
        print("  🎉 Atlas containerizado.")
    else:
        for nome, ok in RESULTADOS:
            if not ok:
                print(f"  🔴 pendente: {nome}")
    print("═" * 56)


if PROJETO is None:
    print("⏸️  defina `PROJETO` acima para rodar a bateria.")
else:
    print(f"▶️  auditando {PROJETO}")

In [ ]:
# ── Bateria 1: os arquivos existem ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    obrigatorios = ["Dockerfile", ".dockerignore", "docker-compose.yml"]
    for nome in obrigatorios:
        checar(f"existe {nome}", (PROJETO / nome).exists())

    ignore = PROJETO / ".dockerignore"
    if ignore.exists():
        linhas = [l.strip() for l in ignore.read_text(encoding="utf-8").splitlines()
                  if l.strip() and not l.startswith("#")]
        checar("🔴 .dockerignore ignora o .env", ".env" in linhas)
        for padrao in [".venv/", ".git/", "__pycache__/"]:
            checar(f".dockerignore ignora {padrao}", padrao in linhas)

    placar()

In [ ]:
# ── Bateria 2: 🔴 o Dockerfile ──
RESULTADOS.clear()

if PROJETO is None or not (PROJETO / "Dockerfile").exists():
    print("⏸️  defina `PROJETO` e crie o Dockerfile")
else:
    texto = (PROJETO / "Dockerfile").read_text(encoding="utf-8")
    passos, achados = analisar_dockerfile(texto)
    graves = mostrar_achados("Dockerfile", achados)
    print()

    checar("🔴 zero problemas graves no Dockerfile", graves == 0, f"{graves} graves")

    froms = [a for _, i, a in passos if i == "FROM"]
    checar("é multi-stage", len(froms) >= 2, f"{len(froms)} estágio(s)")
    checar("🔴 tem USER (não-root)", any(i == "USER" for _, i, _ in passos))
    checar("tem HEALTHCHECK", any(i == "HEALTHCHECK" for _, i, _ in passos))
    checar("CMD/ENTRYPOINT em JSON",
           all(a.strip().startswith("[") for _, i, a in passos
               if i in ("CMD", "ENTRYPOINT")))
    checar("🔴 escuta em 0.0.0.0",
           any("0.0.0.0" in a for _, i, a in passos if i in ("CMD", "ENTRYPOINT")))
    checar("define PYTHONUNBUFFERED",
           any("PYTHONUNBUFFERED" in a for _, i, a in passos if i == "ENV"),
           "sem ele o log não aparece")

    placar()

In [ ]:
# ── Bateria 3: 🔴 o compose (mesclado) ──
RESULTADOS.clear()

if PROJETO is None or not (PROJETO / "docker-compose.yml").exists():
    print("⏸️  defina `PROJETO` e crie o docker-compose.yml")
else:
    # 🔑 mescla base + overrides antes de validar — validar o
    #    fragmento sozinho daria falso positivo
    doc = yaml.safe_load((PROJETO / "docker-compose.yml").read_text(encoding="utf-8"))
    usados = ["docker-compose.yml"]
    for extra in sorted(PROJETO.glob("docker-compose.*.yml")):
        doc = mesclar(doc, yaml.safe_load(extra.read_text(encoding="utf-8")) or {})
        usados.append(extra.name)
    print(f"   mesclando: {' + '.join(usados)}\n")

    _, achados = validar_compose(yaml.safe_dump(doc, sort_keys=False))
    graves = mostrar_achados("compose mesclado", achados)
    print()

    servicos = doc.get("services") or {}
    checar("🔴 zero problemas graves no compose", graves == 0, f"{graves} graves")
    checar("há ao menos 3 serviços", len(servicos) >= 3, f"{len(servicos)}")

    com_hc = [n for n, s in servicos.items() if (s or {}).get("healthcheck")]
    checar("todo serviço com estado tem healthcheck",
           all(any(b in str((s or {}).get("image", "")) for b in COM_ESTADO) <= bool((s or {}).get("healthcheck"))
               for s in servicos.values()),
           f"com healthcheck: {sorted(com_hc)}")

    com_volume = [n for n, s in servicos.items()
                  if any(b in str((s or {}).get("image", "")) for b in COM_ESTADO)
                  and (s or {}).get("volumes")]
    com_estado = [n for n, s in servicos.items()
                  if any(b in str((s or {}).get("image", "")) for b in COM_ESTADO)]
    checar("🔴 todo serviço com estado tem volume",
           set(com_estado) == set(com_volume),
           f"sem volume: {sorted(set(com_estado) - set(com_volume))}")

    usa_healthy = any(
        isinstance((s or {}).get("depends_on"), dict)
        and any((c or {}).get("condition") == "service_healthy"
                for c in (s or {}).get("depends_on", {}).values() if isinstance(c, dict))
        for s in servicos.values())
    checar("🔴 usa condition: service_healthy", usa_healthy)

    placar()

In [ ]:
# ── Bateria 4: 🔒 nenhum segredo em lugar nenhum ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    suspeitos = []
    for arquivo in [*PROJETO.glob("Dockerfile*"), *PROJETO.glob("docker-compose*.yml")]:
        for i, linha in enumerate(arquivo.read_text(encoding="utf-8").splitlines(), 1):
            nua = linha.split("#")[0]
            if SEGREDO.search(nua) and not REFERENCIA.search(nua.split("=")[-1].strip()):
                suspeitos.append(f"{arquivo.name}:{i}")
    checar("🔴 nenhum segredo literal em Dockerfile/compose",
           not suspeitos, str(suspeitos[:3]))

    versionado = PROJETO / ".env"
    gitignore = PROJETO / ".gitignore"
    if gitignore.exists():
        checar("🔴 .env está no .gitignore",
               ".env" in gitignore.read_text(encoding="utf-8"))

    exemplo = PROJETO / ".env.example"
    checar("existe .env.example", exemplo.exists(),
           "documenta as variáveis sem expor valores")

    placar()

In [ ]:
# ── Bateria 5: 🐳 com Docker de verdade ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
elif not DOCKER_LIGADO:
    print("⏸️  Docker não disponível aqui — rode esta bateria na sua máquina:\n")
    print("      docker compose config                     # o YAML final")
    print("      docker compose build")
    print("      docker images atlas-api --format '{{.Size}}'   # < 250 MB")
    print("      docker compose up -d")
    print("      docker compose ps                         # todos healthy")
    print("      docker compose exec api id                # 🔴 NÃO pode ser uid=0")
    print("      docker history atlas-api | grep -i -E 'secret|password'  # vazio")
    print("      time docker compose stop api              # 🔴 < 2 segundos")
    print("      docker compose down && docker compose up -d   # dados preservados")
else:
    checar("docker compose config é válido",
           subprocess.run(["docker", "compose", "config"], cwd=PROJETO,
                          capture_output=True).returncode == 0)
    placar()

> 🎯 **A bateria 5 é a única que exige Docker — e é a que mais importa.**
>
> As quatro primeiras auditam os arquivos: elas pegam quase tudo, e pegam **antes** do build. Mas só rodando você descobre se a imagem realmente sobe, se o healthcheck realmente passa, e se o `docker stop` realmente encerra em menos de 2 segundos.
>
> 💭 É a mesma relação entre o `mypy` e o `pytest` do M04/M07: análise estática pega uma classe de erro; execução pega outra. Nenhuma das duas substitui a outra.

---

## 🎓 Autoavaliação

| # | Consigo… | ✅ |
|---|----------|---|
| 1 | Explicar que um container é um processo, não uma VM | |
| 2 | Nomear os cinco namespaces e o que cada um isola | |
| 3 | Diferenciar namespaces (ver) de cgroups (usar) | |
| 4 | Explicar por que container Linux exige kernel Linux | |
| 5 | Distinguir imagem de container | |
| 6 | Explicar por que `latest` é armadilha | |
| 7 | Dizer quais instruções criam camada | |
| 8 | Explicar por que apagar um arquivo não diminui a imagem | |
| 9 | Ordenar um Dockerfile para maximizar o cache | |
| 10 | Explicar por que dependências vêm antes do código | |
| 11 | Justificar `apt-get update && install` no mesmo `RUN` | |
| 12 | Escrever um `.dockerignore` e dizer por que `.env` é a 1ª linha | |
| 13 | Converter para multi-stage e justificar | |
| 14 | Explicar por que Alpine é armadilha com Python | |
| 15 | Listar as três formas erradas de passar segredo | |
| 16 | Explicar por que `ARG` não esconde nada | |
| 17 | Adicionar usuário não-root e dizer onde vai o `USER` | |
| 18 | Explicar por que `CMD` shell impede o `SIGTERM` | |
| 19 | Explicar por que escutar em `127.0.0.1` quebra o `-p` | |
| 20 | Dizer quando usar volume e quando usar bind mount | |
| 21 | Explicar por que `localhost` não acha o banco no compose | |
| 22 | Explicar por que `depends_on` em lista não basta | |
| 23 | Escrever um healthcheck e justificar o `start_period` | |
| 24 | Diferenciar liveness de readiness | |
| 25 | Ler `Exited (137)` e `Exited (143)` | |
| 26 | Explicar o que `down -v` faz | |
| 27 | Auditar um Dockerfile e um compose sem ajuda | |

**Menos de 21?** Volte às aulas correspondentes antes do Módulo 09.

---

### ➡️ Próximo módulo

**Módulo 09 — Deploy e CI/CD.** *"Subir versão nova é um ritual de risco."*

Você tem a imagem. Agora falta colocá-la num servidor de verdade, atrás de um proxy reverso, com HTTPS, e fazer isso acontecer sozinho a cada `git push` — incluindo as auditorias que você escreveu neste módulo.